# Signature verification - 256x512, reference-2 multi-seed run

Trains the Stage 3 ConvNeXt-Tiny metric-learning encoder and evaluates it on a Kaggle T4.

**Canonical canvas:** `256 x 512` (H x W), grayscale, white background. This notebook does
not use the legacy 384x192 pipeline.

**Dataset.** Attach the cleaned dataset (`manifest_clean.csv` at the root, with
`BHSig260/`, `CEDAR/`, and `SigResize/` beside it). Expected: 20,548 rows (14,401 train /
3,114 val / 3,033 test), 421 signers.

**Experiment:** no augmentation, validation checkpoint selection using **2 medoid references**,
and three seeds: 42, 43, 44. Each checkpoint is also evaluated at 2 and 3 references.
This tests seed variability without changing preprocessing, loss, mining, or dataset
sampling. Fusion is excluded; no target FAR is asserted as a product requirement.

**Settings:** GPU **T4**, **Internet ON** (pretrained ConvNeXt weights), outputs under
`/kaggle/working`.


In [ ]:
# Cell 1 - safely clone or fast-forward the repository.
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/yogeshY0/signautre-256-512.git"
REPO_DIR = Path("/kaggle/working/repo")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"Refusing to overwrite non-git directory: {REPO_DIR}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

%cd /kaggle/working/repo
!git log --oneline -1
!python -m pip install -q timm opencv-python scikit-learn Pillow tqdm


In [ ]:
# Cell 2 - environment check.
import platform
import sys

import cv2
import timm
import torch

print("python :", sys.version.split()[0], platform.platform())
print("torch  :", torch.__version__, "| cuda:", torch.cuda.is_available())
print("gpu    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("timm   :", timm.__version__, "| cv2:", cv2.__version__)
assert torch.cuda.is_available(), "Select the T4 GPU accelerator before running"


In [ ]:
# Cell 3 - locate the cleaned dataset or extract its verified ZIP archive.
from pathlib import Path
from zipfile import ZipFile

PREFERRED_ROOT = Path("/kaggle/input/datasets/yogeshbaral/training-in-256512")

if (PREFERRED_ROOT / "manifest_clean.csv").is_file():
    IMAGE_ROOT = PREFERRED_ROOT
    MANIFEST = IMAGE_ROOT / "manifest_clean.csv"
else:
    candidates = sorted(Path("/kaggle/input").rglob("manifest_clean.csv"))
    if candidates:
        MANIFEST = candidates[0]
        IMAGE_ROOT = MANIFEST.parent
    else:
        archives = sorted(
            Path("/kaggle/input").rglob("training_clean_256x512.zip")
        )
        assert archives, (
            "No manifest_clean.csv or training_clean_256x512.zip under /kaggle/input; "
            "attach the cleaned dataset"
        )
        IMAGE_ROOT = Path("/kaggle/working/training_clean_256x512")
        IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
        with ZipFile(archives[0]) as archive:
            resolved_root = IMAGE_ROOT.resolve()
            for member in archive.infolist():
                target = (IMAGE_ROOT / member.filename).resolve()
                if target != resolved_root and resolved_root not in target.parents:
                    raise ValueError(f"unsafe archive path: {member.filename}")
            archive.extractall(IMAGE_ROOT)
        MANIFEST = IMAGE_ROOT / "manifest_clean.csv"
        print("extracted dataset archive:", archives[0])

assert MANIFEST.is_file(), f"manifest is not a file: {MANIFEST}"
print("manifest  :", MANIFEST)
print("image root:", IMAGE_ROOT)
print("top level :", sorted(path.name for path in IMAGE_ROOT.iterdir()))


In [ ]:
# Cell 4 - counts, dataset mix per split, and path resolution.
import collections
import csv

with MANIFEST.open(encoding="utf-8") as file:
    rows = list(csv.DictReader(file))
print("rows    :", len(rows))
print("splits  :", dict(collections.Counter(row["split"] for row in rows)))
print("signers :", len({row["signer_id"] for row in rows}))
print("datasets:", dict(collections.Counter(row["dataset"] for row in rows)))
print("dataset counts by split:")
for split in ("train", "val", "test"):
    counts = collections.Counter(
        row["dataset"] for row in rows if row["split"] == split
    )
    print(f"  {split}: {dict(sorted(counts.items()))}")
missing = [
    row["image_path"] for row in rows if not (IMAGE_ROOT / row["image_path"]).is_file()
]
print("missing :", len(missing), missing[:3])
assert len(rows) == 20548, f"expected 20548 rows, got {len(rows)}"
assert not missing, "manifest paths do not resolve under IMAGE_ROOT"
print("PASS: manifest and image paths")


In [ ]:
# Cell 5 - run both dataset integrity gates before training.
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "training.verify_data",
        "--manifest",
        str(MANIFEST),
        "--image-root",
        str(IMAGE_ROOT),
        "--skip-augment",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "verification.validate_dataset",
        "--manifest",
        str(MANIFEST),
        "--image-root",
        str(IMAGE_ROOT),
        "--report",
        "/kaggle/working/validate_report.json",
    ],
    check=True,
)
print("PASS: integrity and 256x512 grayscale validation")


## 1. Train: reference 2, no augmentation, three seeds

Each run uses the same data, no augmentation, and the same sampler settings; only the
seed changes. The model checkpoint is selected by validation skilled-forgery EER using
**2 medoid-selected references**. Evaluation later reports both 2- and 3-reference
protocols. This is a three-seed repeat, not a SigResize-specific training intervention.

The trainer prints the configured maximum input tensor shape and estimated tensor memory
before each run. This excludes model activations, gradients, and optimizer state.


In [ ]:
# Cell 6 - shared settings and training command.
import sys

SEEDS = (42, 43, 44)
PHASE2_EPOCHS = 40
PATIENCE = 8
COMMON_ARGS = [
    sys.executable,
    "-u",
    "-m",
    "training.train_siamese",
    "--manifest",
    str(MANIFEST),
    "--image-root",
    str(IMAGE_ROOT),
    "--already-preprocessed",
    "--image_height",
    "256",
    "--image_width",
    "512",
    "--device",
    "cuda",
    "--workers",
    "2",
    "--phase1-epochs",
    "5",
    "--phase2-epochs",
    str(PHASE2_EPOCHS),
    "--patience",
    str(PATIENCE),
    "--select-by",
    "eer",
    "--references",
    "2",
    "--reference-selection",
    "medoid",
]
print("seeds:", SEEDS)
print("common command:", " ".join(COMMON_ARGS))


In [ ]:
# Cell 6b - guard reference availability and signer-disjoint split integrity.
import collections

REFERENCES = 2
genuine_counts = collections.defaultdict(int)
signer_splits = collections.defaultdict(set)
for row in rows:
    signer_splits[row["signer_id"]].add(row["split"])
    if row["is_forgery"].strip().lower() not in ("true", "1", "yes"):
        genuine_counts[row["signer_id"]] += 1

overlapping_signers = {
    signer: sorted(splits) for signer, splits in signer_splits.items() if len(splits) > 1
}
assert not overlapping_signers, f"signers span multiple splits: {overlapping_signers}"
too_small = sorted(
    (signer, genuine_counts[signer], next(iter(signer_splits[signer])))
    for signer in signer_splits
    if next(iter(signer_splits[signer])) in ("val", "validation", "test")
    and genuine_counts[signer] < REFERENCES + 1
)
print("val/test signers with fewer than", REFERENCES + 1, "genuine images:", too_small)
assert not too_small, "some val/test signers cannot support 2 references; fix the data first"
print("PASS: signers are split-disjoint and have enough genuine references")


In [ ]:
# Cell 7 - train the three no-augmentation runs and stream each log.
import subprocess


def run_logged(command, log_path):
    print("\nRunning:", " ".join(command), flush=True)
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        if process.stdout is None:
            raise RuntimeError("training process did not provide an output stream")
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
        return_code = process.wait()
        if return_code:
            raise subprocess.CalledProcessError(return_code, command)


for seed in SEEDS:
    output_dir = Path(f"/kaggle/working/ckpt_seed{seed}")
    log_path = Path(f"/kaggle/working/train_seed{seed}.log")
    command = COMMON_ARGS + [
        "--seed",
        str(seed),
        "--no-augment",
        "--output-dir",
        str(output_dir),
    ]
    run_logged(command, log_path)


In [ ]:
# Cell 7b - verify the saved checkpoint configuration for every seed.
import torch

CHECKPOINTS = {}
for seed in SEEDS:
    checkpoint_path = Path(f"/kaggle/working/ckpt_seed{seed}/best_model.pt")
    assert checkpoint_path.is_file(), f"checkpoint missing for seed {seed}"
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    assert checkpoint["references"] == 2
    assert checkpoint["preprocessing"]["image_height"] == 256
    assert checkpoint["preprocessing"]["image_width"] == 512
    assert checkpoint["augmentation"]["enabled"] is False
    CHECKPOINTS[seed] = str(checkpoint_path)
    print(
        f"seed {seed}: {checkpoint['phase']} epoch {checkpoint['epoch']}, "
        f"validation skilled EER {checkpoint['best_validation_eer_skilled']:.4f}"
    )


## 2. Evaluate each seed on validation and test

Each checkpoint is selected at reference count 2; reports include a 2-vs-3 reference
sweep. Thresholds are chosen on validation signers and applied unchanged to test. Review
per-dataset skilled EER (especially SigResize), random-impostor results, and signer-level
confidence intervals. These development results are not a production FAR guarantee.


In [ ]:
# Cell 8 - evaluate all three checkpoints.
import json
import subprocess

Path("/kaggle/working/reports").mkdir(parents=True, exist_ok=True)
REPORTS = {}
for seed, checkpoint_path in CHECKPOINTS.items():
    report_path = Path(f"/kaggle/working/reports/eval_seed{seed}.json")
    command = [
        sys.executable,
        "-m",
        "training.evaluate",
        "--checkpoint",
        checkpoint_path,
        "--manifest",
        str(MANIFEST),
        "--image-root",
        str(IMAGE_ROOT),
        "--already-preprocessed",
        "--image_height",
        "256",
        "--image_width",
        "512",
        "--references",
        "2",
        "--reference-counts",
        "2,3",
        "--reference-selection",
        "medoid",
        "--device",
        "cuda",
        "--batch-size",
        "32",
        "--workers",
        "2",
        "--target-frr",
        "0.10",
        "--bootstrap",
        "1000",
        "--report",
        str(report_path),
    ]
    subprocess.run(command, check=True)
    REPORTS[seed] = json.loads(report_path.read_text())
print("evaluated seeds:", sorted(REPORTS))


In [ ]:
# Cell 9 - per-seed results, across-seed variation, and reference sweep.
import pandas as pd

result_rows = []
sweep_rows = []
for seed, report in sorted(REPORTS.items()):
    result_rows.append(
        {
            "seed": seed,
            "val_EER": report["validation_eer_skilled"],
            "test_EER": report["test"]["eer_skilled"],
            "test_AUC": report["test"].get("auc_skilled"),
            "SigResize_test_EER": report["per_dataset"]["test"]["SigResize"]["eer_skilled"],
        }
    )
    for refs, block in report["reference_sweep"].items():
        sweep_rows.append(
            {
                "seed": seed,
                "references": int(refs),
                "val_EER": block["validation_eer_skilled"],
                "test_EER": block["test"]["eer_skilled"],
            }
        )

results = pd.DataFrame(result_rows).set_index("seed")
print(results.round(4).to_string())
print("\nAcross-seed mean and standard deviation:")
print(results.agg(["mean", "std"]).round(4).to_string())
print("\nReference-count sweep:")
print(
    pd.DataFrame(sweep_rows)
    .sort_values(["seed", "references"])
    .round(4)
    .to_string(index=False)
)


## 3. Per-corpus audit

Run the same 2-reference medoid protocol for each seed. The audit is read-only and helps
show whether source-level behavior is consistent across initializations.


In [ ]:
# Cell 10 - per-dataset audit for each seed.
import subprocess

for seed, checkpoint_path in CHECKPOINTS.items():
    subprocess.run(
        [
            sys.executable,
            "-u",
            "-m",
            "training.audit_embeddings",
            "--checkpoint",
            checkpoint_path,
            "--manifest",
            str(MANIFEST),
            "--image-root",
            str(IMAGE_ROOT),
            "--already-preprocessed",
            "--image_height",
            "256",
            "--image_width",
            "512",
            "--references",
            "2",
            "--reference-selection",
            "medoid",
            "--device",
            "cuda",
            "--batch-size",
            "32",
            "--workers",
            "2",
            "--max-random-queries",
            "2000",
            "--hist-dir",
            f"/kaggle/working/audit_seed{seed}",
        ],
        check=True,
    )


## 4. Package the artifacts

`/kaggle/working` is what Kaggle offers for download when the session ends.


In [ ]:
# Cell 11 - package all three checkpoints, logs, reports, and per-seed audits.
import subprocess

artifact_paths = ["reports", "validate_report.json"]
for seed in SEEDS:
    artifact_paths.extend(
        [
            f"ckpt_seed{seed}/best_model.pt",
            f"train_seed{seed}.log",
            f"audit_seed{seed}",
        ]
    )
subprocess.run(
    [
        "tar",
        "-czf",
        "/kaggle/working/artifacts_ref2_seeds.tar.gz",
        "-C",
        "/kaggle/working",
        *artifact_paths,
    ],
    check=True,
)
!ls -lh /kaggle/working/artifacts_ref2_seeds.tar.gz


## Reading the results

* Compare `test_EER` and `SigResize_test_EER` across seeds; report the mean and standard
  deviation, not only the best seed.
* `reference_sweep` shows 2 vs. 3 references for each checkpoint. The checkpoint itself
  was selected using 2 references.
* `per_dataset.test.<corpus>.eer_skilled` is the primary source-level view. Review
  SigResize errors and labels before changing images or adding source-specific transforms.
* The 10% validation-FRR operating point is diagnostic, not a product requirement.
* Do not quote metrics from a smoke run or use these same test signers to tune the model.
* Download `artifacts_ref2_seeds.tar.gz` before the Kaggle session ends.

**Canonical canvas:** 256 x 512. Do not merge the old 384 x 192 upstream into this branch.
